[Project Stone]
- 돌 분류 프로젝트

In [4]:
import sys
import torch
import os

print("--- Environment Check ---")
print(f"Python Executable: {sys.executable}") # 현재 사용 중인 파이썬 실행 파일 경로
print(f"Python Version: {sys.version}")      # 현재 사용 중인 파이썬 버전
print(f"PyTorch Version: {torch.__version__}") # 현재 로드된 PyTorch 버전
print(f"PyTorch CUDA Build: {torch.version.cuda if hasattr(torch.version, 'cuda') else 'N/A'}") # PyTorch가 빌드된 CUDA 버전
print(f"CUDA Available: {torch.cuda.is_available()}") # CUDA 사용 가능 여부
if torch.cuda.is_available():
    print(f"CUDA Version (Runtime): {torch.version.cuda}") # PyTorch가 인식하는 런타임 CUDA 버전
    print(f"Device Name: {torch.cuda.get_device_name(0)}") # GPU 이름
    print(f"Device Compute Capability: {torch.cuda.get_device_capability(0)}") # Compute Capability 확인
print("-" * 25)

--- Environment Check ---
Python Executable: c:\Users\user\anaconda3\envs\tch26_cuda128\python.exe
Python Version: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
PyTorch Version: 2.7.0+cu128
PyTorch CUDA Build: 12.8
CUDA Available: True
CUDA Version (Runtime): 12.8
Device Name: NVIDIA GeForce RTX 5090
Device Compute Capability: (12, 0)
-------------------------


In [ ]:
import pandas as pd
import numpy as np

In [6]:
numlist = os.listdir('./_data/open/train')
trainDIR = './_data/open/train'
sum = 0

for i in numlist:
    a = (len(os.listdir('./_data/open/train/'+i)))
    print(i, a)
    sum += a
print(sum)

Andesite 43802
Basalt 26810
Etc 15935
Gneiss 73914
Granite 92923
Mud_Sandstone 89467
Weathered_Rock 37169
380020


In [7]:
numlist = os.listdir('./_data/open/test')
testDIR = './_data/open/test'
len(numlist)

95006

In [8]:
## 작업순서
## 데이테 셋 만들기 
## 데이터 로더 만들기
## 폴더별로 되있으니까 imgeafolder사용하면 될듯?


In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from torch.utils.data import Dataset, DataLoader  # Pytorch의 데이터셋 관련
from torchvision import transforms  # 전처리모듈
from torchvision.datasets import ImageFolder

from PIL import Image


In [10]:
TRANSFORM = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
])



In [11]:
classes = ['Andesite', 'Basalt', 'Gneiss', 'Granite', 'Mud_Sandstone', 'Weathered_Rock']
classes2idx = {x:idx+1 for idx,x in enumerate(classes)}
classes2idx['Etc']= 0
classes2idx

{'Andesite': 1,
 'Basalt': 2,
 'Gneiss': 3,
 'Granite': 4,
 'Mud_Sandstone': 5,
 'Weathered_Rock': 6,
 'Etc': 0}

In [12]:

class CustomImageFolder(ImageFolder):
    def __init__(self, root, transform=None, classes2idx=classes2idx):
        self.classes2idx = classes2idx
        self.class_to_idx = self.classes2idx
        self.classes = list(classes2idx.keys())
        super().__init__(root, transform)

    def find_classes(self, directory):
        # 이 함수가 자동 클래스 탐색을 담당하는데, 우리가 원하는 대로 덮어씀
        return self.classes, self.classes2idx

In [13]:
trainDS = CustomImageFolder(trainDIR, transform=TRANSFORM, classes2idx=classes2idx)

In [14]:
for a,b in trainDS:
    print(a,b)
    break

tensor([[[-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         ...,
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179]],

        [[-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         ...,
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357]],

        [[-1.8044, -1.8044, -1.8044,  ..., -1.8044, -1.8044, -1.8044],
         [-1.8044, -1.8044, -1.8044,  ..., -1

In [15]:
print('train', len(trainDS))
print(trainDS.classes)
trainDS.class_to_idx
idx2class = {values:key for key, values in classes2idx.items()}
idx2class

train 380020
['Andesite', 'Basalt', 'Gneiss', 'Granite', 'Mud_Sandstone', 'Weathered_Rock', 'Etc']


{1: 'Andesite',
 2: 'Basalt',
 3: 'Gneiss',
 4: 'Granite',
 5: 'Mud_Sandstone',
 6: 'Weathered_Rock',
 0: 'Etc'}

In [16]:
trainDS[111100][0].shape

torch.Size([3, 224, 224])

In [17]:
cnum = 0
for (a, b) in trainDS:
    cnum += 1
    print(a[0].shape, idx2class[b])
    if cnum == 10: break
    

torch.Size([224, 224]) Andesite
torch.Size([224, 224]) Andesite
torch.Size([224, 224]) Andesite
torch.Size([224, 224]) Andesite
torch.Size([224, 224]) Andesite
torch.Size([224, 224]) Andesite
torch.Size([224, 224]) Andesite
torch.Size([224, 224]) Andesite
torch.Size([224, 224]) Andesite
torch.Size([224, 224]) Andesite


In [18]:
from torch.utils.data import random_split

# 전체 데이터 수
total_size = len(trainDS)
train_size = int(0.8 * total_size)
valid_size = total_size - train_size

# 무작위로 train/valid 분리
trainDS, validDS = random_split(trainDS, [train_size, valid_size])

In [19]:
print(len(trainDS))
print(len(validDS))

304016
76004


In [20]:
imgTS, label = trainDS[0]   ## __getitem__(index)
print(imgTS.shape, label)


torch.Size([3, 224, 224]) 3


In [21]:
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches

In [22]:
# plt.imshow(imgTS.permute(1,2,0))
# plt.title(idx2class[label])
# plt.show()

In [23]:
def collator(batch):
    images, labels = zip(*batch)  # 튜플 of Tensors
    images = torch.stack(images)  # → Tensor of shape [B, C, H, W]
    labels = torch.tensor(labels) # → Tensor of shape [B]
    return images, labels

BATCH_SIZE = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 0.0001
DEVICE

'cuda'

In [24]:
trainDL = DataLoader(
    trainDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)
validDL = DataLoader(
    validDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)

In [25]:
for a, b in trainDL:
    print((type(a)),(type(b)))
    break

<class 'torch.Tensor'> <class 'torch.Tensor'>


In [26]:
from torchvision import models
from torchvision import ops
from torchvision.models.detection import rpn

num_classes = len(classes2idx)
num_classes

7

In [ ]:
# ✅ 모델: ResNet101 + 마지막 fc 교체
model = models.resnet101(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)


c:\Users\user\anaconda3\envs\tch26_cuda128\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\user\anaconda3\envs\tch26_cuda128\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [28]:
from torch import optim
from tqdm import tqdm
# ✅ 손실함수, 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


In [29]:
DEVICE

'cuda'

In [30]:
torch.__version__

'2.7.0+cu128'

In [31]:
# %conda install pytorch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 pytorch-cuda=12.4 -c pytorch -c nvidia

In [32]:
# import torch
# import torch.nn as nn
# from torch.utils.data import DataLoader
# from torchvision import models # 모델 정의를 위해 필요
# from torch import optim # 옵티마이저 정의를 위해 필요
# import os

# # --- 기본 설정 (기존 코드에서 가져옴) ---
# # DEVICE 설정 (필수)
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Using device: {DEVICE}")

# # 모델 정의 및 DEVICE로 이동 (필수)
# # 예시: ResNet101 사용 (실제 사용하는 모델로 대체)
# num_classes = 7 # 예시 클래스 수 (실제 값으로 변경)
# model = models.resnet101(weights='DEFAULT') # 또는 pretrained=True
# model.fc = nn.Linear(model.fc.in_features, num_classes)
# model = model.to(DEVICE)
# print("Model moved to DEVICE.")

# # 손실 함수 정의 (필수)
# criterion = nn.CrossEntropyLoss()

# # 옵티마이저 정의 (필수)
# LR = 0.0001 # 예시 학습률
# optimizer = optim.Adam(model.parameters(), lr=LR)

# # 데이터 로더 정의 (trainDL 하나만 필요)
# # trainDS, BATCH_SIZE, collator는 이미 정의되어 있다고 가정
# # 예시: trainDL = DataLoader(trainDS, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
# # --- 이 부분은 실행 환경에 맞게 trainDL을 준비해야 합니다 ---
# # 아래 코드는 trainDL이 이미 존재한다고 가정하고 진행합니다.
# # if 'trainDL' not in locals():
# #     print("Error: trainDL is not defined. Please define the DataLoader first.")
# #     exit()

# # (선택 사항) CUDA 에러 디버깅을 위한 환경 변수 설정
# # 에러 발생 위치를 더 명확하게 하지만, 근본 원인을 해결하지는 않음
# # os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

# # --- 단순화된 테스트 코드 ---
# print("\n--- Starting Simplified CUDA Test ---")

# try:
#     # 1. DataLoader에서 딱 한 개의 배치만 가져오기
#     print("1. Fetching a single batch from DataLoader...")
#     # iter()를 사용하여 이터레이터 생성 후 next()로 하나 가져옴
#     data_iter = iter(trainDL)
#     images, labels = next(data_iter)
#     print(f"   Batch fetched: images shape {images.shape}, labels shape {labels.shape}")

#     # 2. 가져온 배치를 DEVICE로 이동
#     print(f"2. Moving batch to {DEVICE}...")
#     images = images.to(DEVICE)
#     labels = labels.to(DEVICE)
#     # 이동 확인 (선택적)
#     print(f"   Data device: images on {images.device}, labels on {labels.device}")

#     # 3. 모델을 평가 모드가 아닌 학습 모드로 설정 (BatchNorm 등에 영향)
#     model.train()
#     print("3. Set model to train() mode.")

#     # 4. 옵티마이저 그래디언트 초기화
#     optimizer.zero_grad()
#     print("4. Cleared optimizer gradients (zero_grad).")

#     # 5. 모델 순전파 (Forward pass)
#     print("5. Performing forward pass (model(images))...")
#     outputs = model(images)
#     print(f"   Forward pass successful. Output shape: {outputs.shape}, Output device: {outputs.device}")

#     # 6. 손실 계산
#     print("6. Calculating loss (criterion(outputs, labels))...")
#     loss = criterion(outputs, labels)
#     print(f"   Loss calculated: {loss.item():.4f}, Loss device: {loss.device}")

#     # 7. 역전파 (Backward pass)
#     print("7. Performing backward pass (loss.backward())...")
#     loss.backward()
#     print("   Backward pass successful.")

#     # 8. 옵티마이저 파라미터 업데이트
#     print("8. Updating model parameters (optimizer.step())...")
#     optimizer.step()
#     print("   Optimizer step successful.")

#     print("\n--- Simplified CUDA Test Completed Successfully! ---")

# except StopIteration:
#     print("\nError: DataLoader is empty. Cannot fetch a batch.")
# except RuntimeError as e:
#     print(f"\n--- CUDA Runtime Error Occurred! ---")
#     print(f"Error Message: {e}")
#     # 스택 트레이스는 자동으로 출력됩니다. 에러 발생 지점을 확인하세요.
# except Exception as e:
#     print(f"\n--- An Unexpected Error Occurred! ---")
#     print(f"Error Type: {type(e).__name__}")
#     print(f"Error Message: {e}")

In [ ]:
# %pip install psutil
import psutil
import subprocess
from tqdm import tqdm

def get_gpu_usage():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,nounits,noheader"]
        )
        result = result.decode("utf-8").strip().split("\n")[0]
        gpu_util, mem_used, mem_total = map(int, result.split(", "))
        return gpu_util, mem_used, mem_total
    except Exception:
        return None, None, None

In [7]:
import time

In [47]:
now = time.localtime()
ct = time.strftime("%y.%m.%d %H:%M:%S",now)
ct

'25.04.30 11:23:43'

In [35]:
EPOCH = 5
best_val_loss = float('inf')
patience = 3
patience_counter = 0

train_metrics = []
val_metrics = []

for epoch in range(EPOCH):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    epoch_train_cpu = []
    epoch_train_gpu = []
    epoch_train_loss = []
    epoch_train_acc = []

    train_loop = tqdm(trainDL, desc=f"[Epoch {epoch+1}] Training")
    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        acc = 100. * correct / total

        # Metric 측정
        cpu_usage = psutil.cpu_percent(interval=None)
        gpu_util, mem_used, mem_total = get_gpu_usage()

        # 저장
        epoch_train_cpu.append(cpu_usage)
        epoch_train_gpu.append(gpu_util if gpu_util is not None else 0)
        epoch_train_loss.append(loss.item())
        epoch_train_acc.append(acc)

        train_loop.set_postfix(loss=loss.item(), acc=acc, cpu=f"{cpu_usage}%", gpu=f"{gpu_util}%")

    train_metrics.append({
        "cpu": epoch_train_cpu,
        "gpu": epoch_train_gpu,
        "loss": epoch_train_loss,
        "acc": epoch_train_acc
    })

    train_acc = 100. * correct / total
    train_loss = running_loss / len(trainDL)

    # ----------- Validation -----------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    epoch_val_cpu = []
    epoch_val_gpu = []
    epoch_val_loss = []
    epoch_val_acc = []

    val_loop = tqdm(validDL, desc=f"[Epoch {epoch+1}] Validation")
    with torch.no_grad():
        for images, labels in val_loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)

            acc = 100. * val_correct / val_total

            cpu_usage = psutil.cpu_percent(interval=None)
            gpu_util, mem_used, mem_total = get_gpu_usage()

            # 저장
            epoch_val_cpu.append(cpu_usage)
            epoch_val_gpu.append(gpu_util if gpu_util is not None else 0)
            epoch_val_loss.append(loss.item())
            epoch_val_acc.append(acc)

            val_loop.set_postfix(loss=loss.item(), acc=acc, cpu=f"{cpu_usage}%", gpu=f"{gpu_util}%")

    val_metrics.append({
        "cpu": epoch_val_cpu,
        "gpu": epoch_val_gpu,
        "loss": epoch_val_loss,
        "acc": epoch_val_acc
    })

    val_acc = 100. * val_correct / val_total
    val_loss_avg = val_loss / len(validDL)

    if val_loss_avg < best_val_loss and val_loss_avg < 2:
        best_val_loss = val_loss_avg
        torch.save(model.state_dict(), f"./_model/{epoch}_vl{val_loss_avg:.2f}.pth")

    print(f"\n[Epoch {epoch+1}] Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"                    Valid Loss: {val_loss_avg:.4f}, Valid Acc: {val_acc:.2f}%")

# -------------------------------
# 모든 metric을 담은 리스트
# -------------------------------
# 예: train_metrics[0]["loss"] → 첫 번째 에포크의 모든 배치 loss
#     val_metrics[0]["gpu"]  → 첫 번째 에포크의 모든 배치 GPU 사용률

[Epoch 1] Validation: 100%|██████████| 760/760 [05:16<00:00,  2.40it/s, acc=84.9, cpu=9.4%, gpu=23%, loss=0.467] 



[Epoch 1] Train Loss: 0.4976, Train Acc: 82.54%
                    Valid Loss: 0.4308, Valid Acc: 84.89%


[Epoch 2] Validation: 100%|██████████| 760/760 [05:52<00:00,  2.15it/s, acc=87.2, cpu=6.6%, gpu=36%, loss=0.316] 



[Epoch 2] Train Loss: 0.3925, Train Acc: 86.27%
                    Valid Loss: 0.3659, Valid Acc: 87.23%


[Epoch 3] Validation: 100%|██████████| 760/760 [05:37<00:00,  2.25it/s, acc=88.4, cpu=13.0%, gpu=0%, loss=0.419] 



[Epoch 3] Train Loss: 0.3449, Train Acc: 87.91%
                    Valid Loss: 0.3310, Valid Acc: 88.36%


[Epoch 4] Validation: 100%|██████████| 760/760 [05:52<00:00,  2.16it/s, acc=88.7, cpu=11.1%, gpu=0%, loss=0.313] 



[Epoch 4] Train Loss: 0.3104, Train Acc: 89.02%
                    Valid Loss: 0.3237, Valid Acc: 88.67%


[Epoch 5] Validation: 100%|██████████| 760/760 [05:54<00:00,  2.14it/s, acc=89.6, cpu=12.9%, gpu=0%, loss=0.249] 



[Epoch 5] Train Loss: 0.2822, Train Acc: 89.97%
                    Valid Loss: 0.2945, Valid Acc: 89.59%


In [ ]:

EPOCH = 5
best_val_loss = float('inf')
patience = 3
patience_counter = 0

# 에포크별 '평균' 지표를 저장할 리스트
train_metrics = []
val_metrics = []

# psutil.cpu_percent() 초기 호출 (첫 호출 시 의미 없는 값 반환 방지)
psutil.cpu_percent(interval=None)
time.sleep(0.1) # CPU 사용률 측정을 위한 짧은 대기

for epoch in range(EPOCH):
    # ----------- Training -----------
    model.train()
    # --- 에포크 누적 변수 초기화 ---
    epoch_train_loss_sum = 0.0
    epoch_train_correct = 0
    epoch_train_total = 0
    epoch_train_cpu_sum = 0.0
    epoch_train_gpu_sum = 0.0
    train_batches = 0 # 실제 처리된 배치 수 카운트

    train_loop = tqdm(trainDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Training", leave=False)
    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # --- 배치 결과 누적 ---
        epoch_train_loss_sum += loss.item()
        _, predicted = outputs.max(1)
        epoch_train_correct += predicted.eq(labels).sum().item()
        epoch_train_total += labels.size(0)
        train_batches += 1

        # --- 시스템 지표 측정 및 누적 ---
        cpu_usage = psutil.cpu_percent(interval=None)
        # interval=None 은 마지막 호출 이후의 평균을 반환하므로 루프 내에서 반복 호출 시 적합
        gpu_util, _, _ = get_gpu_usage() # 메모리 사용량은 여기선 저장 안 함

        epoch_train_cpu_sum += cpu_usage
        epoch_train_gpu_sum += gpu_util if gpu_util is not None else 0

        # --- tqdm 진행률 표시줄 업데이트 (현재 '배치' 손실과 '누적' 정확도 표시) ---
        # 원본 코드의 누적 정확도 표시 방식 유지
        current_cumulative_acc = 100. * epoch_train_correct / epoch_train_total
        train_loop.set_postfix(loss=loss.item(), # 현재 배치 손실
                               acc=f"{current_cumulative_acc:.2f}%", # 현재까지 누적 정확도
                               cpu=f"{cpu_usage:.1f}%",
                               gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")

    # --- 에포크 평균 계산 ---
    # train_batches가 0인 경우 방지
    if train_batches > 0:
        avg_train_loss = epoch_train_loss_sum / train_batches
        avg_train_acc = 100. * epoch_train_correct / epoch_train_total
        avg_train_cpu = epoch_train_cpu_sum / train_batches
        avg_train_gpu = epoch_train_gpu_sum / train_batches
    else:
        avg_train_loss, avg_train_acc, avg_train_cpu, avg_train_gpu = 0, 0, 0, 0

    # --- 에포크 평균 지표 저장 ---
    train_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_train_loss,
        "accuracy": avg_train_acc,
        "avg_cpu_percent": avg_train_cpu,
        "avg_gpu_percent": avg_train_gpu
    })

    # ----------- Validation -----------
    model.eval()
    # --- 에포크 누적 변수 초기화 ---
    epoch_val_loss_sum = 0.0
    epoch_val_correct = 0
    epoch_val_total = 0
    epoch_val_cpu_sum = 0.0
    epoch_val_gpu_sum = 0.0
    val_batches = 0 # 실제 처리된 배치 수 카운트

    val_loop = tqdm(validDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Validation", leave=False)
    with torch.no_grad():
        for images, labels in val_loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            # --- 배치 결과 누적 ---
            epoch_val_loss_sum += loss.item()
            _, predicted = outputs.max(1)
            epoch_val_correct += predicted.eq(labels).sum().item()
            epoch_val_total += labels.size(0)
            val_batches += 1

            # --- 시스템 지표 측정 및 누적 ---
            cpu_usage = psutil.cpu_percent(interval=None)
            gpu_util, _, _ = get_gpu_usage()

            epoch_val_cpu_sum += cpu_usage
            epoch_val_gpu_sum += gpu_util if gpu_util is not None else 0

            # --- tqdm 진행률 표시줄 업데이트 (현재 '배치' 손실과 '누적' 정확도 표시) ---
            current_cumulative_acc = 100. * epoch_val_correct / epoch_val_total
            val_loop.set_postfix(loss=loss.item(), # 현재 배치 손실
                                 acc=f"{current_cumulative_acc:.2f}%", # 현재까지 누적 정확도
                                 cpu=f"{cpu_usage:.1f}%",
                                 gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")

    # --- 에포크 평균 계산 ---
    # val_batches가 0인 경우 방지
    if val_batches > 0:
        avg_val_loss = epoch_val_loss_sum / val_batches
        avg_val_acc = 100. * epoch_val_correct / epoch_val_total
        avg_val_cpu = epoch_val_cpu_sum / val_batches
        avg_val_gpu = epoch_val_gpu_sum / val_batches
    else:
        avg_val_loss, avg_val_acc, avg_val_cpu, avg_val_gpu = 0, 0, 0, 0

    # --- 에포크 평균 지표 저장 ---
    val_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_val_loss,
        "accuracy": avg_val_acc,
        "avg_cpu_percent": avg_val_cpu,
        "avg_gpu_percent": avg_val_gpu
    })

    # --- 결과 출력 및 모델 저장 로직 (Early Stopping 포함 가능) ---
    print(f"\n[Epoch {epoch+1}/{EPOCH}]")
    print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.2f}% | Avg CPU: {avg_train_cpu:.1f}%, Avg GPU: {avg_train_gpu:.1f}%")
    print(f"  Valid Loss: {avg_val_loss:.4f}, Valid Acc: {avg_val_acc:.2f}% | Avg CPU: {avg_val_cpu:.1f}%, Avg GPU: {avg_val_gpu:.1f}%")

    # Early Stopping 및 모델 저장 (평균 검증 손실 사용)
    # 원본 코드의 저장 조건 유지 (val_loss_avg < 2 부분은 필요에 따라 제거/수정)
    if avg_val_loss < best_val_loss : # and avg_val_loss < 2:
        print(f"  Validation loss improved ({best_val_loss:.4f} --> {avg_val_loss:.4f}). Saving model...")
        best_val_loss = avg_val_loss
        # 모델 저장 경로 및 이름 확인 필요 (_model 폴더가 있어야 함)
        # try:
        #     torch.save(model.state_dict(), f"./_model/epoch{epoch+1}_best_vl{avg_val_loss:.4f}.pth")
        # except FileNotFoundError:
        #     print("  Warning: './_model' directory not found. Skipping model saving.")
        patience_counter = 0 # 개선되었으므로 카운터 초기화
    else:
        patience_counter += 1
        print(f"  Validation loss did not improve from {best_val_loss:.4f}. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"  Early stopping triggered after {epoch + 1} epochs.")
            break # 학습 중단

# 학습 완료 후 저장된 지표 확인 (예시)
print("\n--- Training Metrics (Epoch Averages) ---")
for i, metrics in enumerate(train_metrics):
    print(f"Epoch {metrics['epoch']}: {metrics}")

print("\n--- Validation Metrics (Epoch Averages) ---")
for i, metrics in enumerate(val_metrics):
    print(f"Epoch {metrics['epoch']}: {metrics}")

# -------------------------------
# train_metrics 와 val_metrics 리스트에는 이제 각 에포크의 평균 지표가
# 딕셔너리 형태로 저장되어 있습니다.
# 예: train_metrics[0] -> 첫 번째 에포크의 평균 {loss, accuracy, avg_cpu_percent, avg_gpu_percent}
# -------------------------------



In [ ]:
# 이제 이 데이터를 pandas DataFrame으로 변환하여 CSV로 저장할 수 있습니다.
# 예시:
train_df = pd.DataFrame(train_metrics)
val_df = pd.DataFrame(val_metrics)
train_df.to_csv("train_metrics.csv", index=False)
val_df.to_csv("val_metrics.csv", index=False)

NameError: name 'train_metrics' is not defined

In [37]:
import torch
print(torch.__version__)             # ex) 2.4.1
print(torch.version.cuda)           # ex) 12.4
print(torch.cuda.get_device_name(0)) # ex) NVIDIA RTX 5090
print(torch.cuda.is_available()) 

2.7.0+cu128
12.8
NVIDIA GeForce RTX 5090
True


In [38]:
sampleDF = pd.DataFrame(pd.read_csv('./_data/open/sample_submission.csv'))
print(sampleDF.head())
testDF = pd.DataFrame(pd.read_csv('./_data/open/test.csv'))
print(testDF.head())

           ID rock_type
0  TEST_00000       Etc
1  TEST_00001       Etc
2  TEST_00002       Etc
3  TEST_00003       Etc
4  TEST_00004       Etc
           ID               img_path
0  TEST_00000  ./test/TEST_00000.jpg
1  TEST_00001  ./test/TEST_00001.jpg
2  TEST_00002  ./test/TEST_00002.jpg
3  TEST_00003  ./test/TEST_00003.jpg
4  TEST_00004  ./test/TEST_00004.jpg


In [39]:
class TestImageDataset(Dataset):
    def __init__(self, csv_df, image_root, transform=None):
        self.df = csv_df
        self.image_root = image_root  # 예: './_data/open/test/'
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0]
        img_path = self.df.iloc[idx, 1]
        img_path = os.path.join(self.image_root, img_path)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name


In [40]:
TRANSFORM_T = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
])



In [41]:
testDS = TestImageDataset(testDF, image_root="./_data/open/", transform=TRANSFORM_T)
testDL = DataLoader(testDS, batch_size=32, shuffle=False)

In [42]:
for a, b in testDS:
    print(a,b)
    break

tensor([[[-0.5253, -0.4739, -0.6281,  ..., -0.9534, -1.0904, -1.2274],
         [-0.6281, -0.8335, -0.6965,  ..., -0.8507, -0.7993, -1.0219],
         [-0.9877, -0.6452, -0.7479,  ..., -1.0562, -0.9705, -0.9705],
         ...,
         [-0.7822, -1.0562, -1.2274,  ..., -1.1247, -1.1760, -1.2617],
         [-1.0048, -1.1418, -0.9534,  ..., -1.2617, -1.2103, -1.0904],
         [-0.9363, -0.9192, -0.8164,  ..., -1.1247, -1.1760, -1.1932]],

        [[-0.2500, -0.1975, -0.3725,  ..., -0.7402, -0.8803, -1.0203],
         [-0.3725, -0.5826, -0.4426,  ..., -0.6352, -0.5826, -0.8102],
         [-0.7227, -0.3725, -0.4776,  ..., -0.8452, -0.7577, -0.7577],
         ...,
         [-0.4776, -0.7577, -0.9328,  ..., -0.8978, -0.9503, -1.0378],
         [-0.7052, -0.8452, -0.6527,  ..., -1.0378, -0.9853, -0.8627],
         [-0.6352, -0.6176, -0.5126,  ..., -0.8978, -0.9503, -0.9678]],

        [[-0.0441,  0.0431, -0.1138,  ..., -0.3055, -0.4450, -0.5844],
         [-0.1312, -0.3230, -0.1835,  ..., -0

In [ ]:
model.eval()
predictions = []

with torch.no_grad():
    count = 0
    for images, filenames in testDL:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        
        for fname, pred in zip(filenames, preds):
            predictions.append((fname, idx2class[pred.item()]))
        count += 1
        # if count ==3: break

In [44]:
sampleDF.head()

,ID,rock_type
0,TEST_00000,Etc
1,TEST_00001,Etc
2,TEST_00002,Etc
3,TEST_00003,Etc
4,TEST_00004,Etc


In [ ]:
len(predictions)

[('TEST_00000', 'Mud_Sandstone'),
 ('TEST_00001', 'Mud_Sandstone'),
 ('TEST_00002', 'Mud_Sandstone'),
 ('TEST_00003', 'Granite'),
 ('TEST_00004', 'Granite'),
 ('TEST_00005', 'Granite'),
 ('TEST_00006', 'Granite'),
 ('TEST_00007', 'Granite'),
 ('TEST_00008', 'Granite'),
 ('TEST_00009', 'Granite'),
 ('TEST_00010', 'Granite'),
 ('TEST_00011', 'Granite'),
 ('TEST_00012', 'Granite'),
 ('TEST_00013', 'Granite'),
 ('TEST_00014', 'Etc'),
 ('TEST_00015', 'Etc'),
 ('TEST_00016', 'Granite'),
 ('TEST_00017', 'Granite'),
 ('TEST_00018', 'Granite'),
 ('TEST_00019', 'Etc'),
 ('TEST_00020', 'Granite'),
 ('TEST_00021', 'Granite'),
 ('TEST_00022', 'Granite'),
 ('TEST_00023', 'Granite'),
 ('TEST_00024', 'Granite'),
 ('TEST_00025', 'Granite'),
 ('TEST_00026', 'Granite'),
 ('TEST_00027', 'Granite'),
 ('TEST_00028', 'Granite'),
 ('TEST_00029', 'Granite'),
 ('TEST_00030', 'Granite'),
 ('TEST_00031', 'Granite'),
 ('TEST_00032', 'Granite'),
 ('TEST_00033', 'Granite'),
 ('TEST_00034', 'Granite'),
 ('TEST_00035'

In [48]:
sampleDF.shape[0]
len(predictions)

96

In [47]:
# for i in range(sampleDF.shape[0]):
for i in range(95006):
    if sampleDF.loc[i,'ID'] == predictions[i][0]:
        sampleDF.loc[i,'rock_type'] = predictions[i][1]
    else:
        continue
sampleDF.head(20)

IndexError: list index out of range

In [ ]:
sampleDF.to_csv('./_data/open/sample_submission_answer.csv')